In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from scipy import stats
from scipy.stats import spearmanr
import os
from bisect import bisect
from collections import defaultdict
from typing import Dict, Tuple, Optional, List

# Print versions
print(f"NumPy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")

# Set numpy print options
np.set_printoptions(threshold=np.inf)

# Publication-quality matplotlib settings
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 10,
    'figure.titlesize': 18,
    'font.family': 'serif',
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight'
})

# Device configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES = 10

print(f"Device: {DEVICE}")

# Create output directories
os.makedirs('figures/accuracy', exist_ok=True)
os.makedirs('figures/fdr_curves', exist_ok=True)
os.makedirs('figures/mano', exist_ok=True)
os.makedirs('figures/comparison', exist_ok=True)

print("✓ Setup complete")

NumPy version: 1.24.3
PyTorch version: 2.0.0+cu117
Device: cuda
✓ Setup complete


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import numpy as np
from datetime import date
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binom
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests
from bisect import bisect
import os

from data_processing.score_feature_dataset import ScoreFeatureDataset, create_score_feature_dataset, create_score_feature_dataset_bcss
from data_processing.negative_scores_pool import collect_negative_scores
from flows.separate_flows import SeparateClassFlows

ModuleNotFoundError: No module named 'data_processing'

In [2]:
!pip install POT

Defaulting to user installation because normal site-packages is not writeable
ERROR: Could not find a version that satisfies the requirement POT (from versions: none)
ERROR: No matching distribution found for POT


In [3]:
def negentropy(logits):
    """Compute negentropy (negative entropy)."""
    if isinstance(logits, np.ndarray):
        logits = torch.from_numpy(logits).float()
    probs = torch.softmax(logits, dim=1)
    entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=1)
    max_entropy = np.log(logits.shape[1])
    return max_entropy - entropy


def calibration_temp(logits, labels, num_bins=15):
    """Find optimal temperature for calibration."""
    if isinstance(logits, np.ndarray):
        logits = torch.from_numpy(logits).float()
    if isinstance(labels, np.ndarray):
        labels = torch.from_numpy(labels).long()
    
    temps = torch.linspace(0.1, 5.0, 50)
    best_temp = 1.0
    best_ece = float('inf')
    
    for temp in temps:
        scaled_probs = torch.softmax(logits / temp, dim=1)
        confidences, predictions = scaled_probs.max(dim=1)
        accuracies = (predictions == labels).float()
        
        bin_boundaries = torch.linspace(0, 1, num_bins + 1)
        ece = 0.0
        
        for i in range(num_bins):
            mask = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
            if mask.sum() > 0:
                bin_conf = confidences[mask].mean()
                bin_acc = accuracies[mask].mean()
                ece += mask.float().mean() * torch.abs(bin_conf - bin_acc)
        
        if ece < best_ece:
            best_ece = ece
            best_temp = temp.item()
    
    return best_temp


def _to_tensor(x):
    """Convert numpy array or tensor to torch tensor."""
    if isinstance(x, np.ndarray):
        return torch.from_numpy(x).float()
    return x


def predict_ATC_maxconf(source_logits, source_labels, target_logits):
    """Average Threshold Confidence with max confidence."""
    source_logits = _to_tensor(source_logits)
    source_labels = _to_tensor(source_labels).long()
    target_logits = _to_tensor(target_logits)
    
    source_scores = torch.softmax(source_logits, dim=1).amax(1)
    target_scores = torch.softmax(target_logits, dim=1).amax(1)
    sorted_source_scores, _ = torch.sort(source_scores)
    threshold = sorted_source_scores[-(source_logits.argmax(1) == source_labels).sum()]
    estimate = (target_scores > threshold).float().mean().item()
    return estimate


def predict_ATC_negent(source_logits, source_labels, target_logits):
    """Average Threshold Confidence with negentropy."""
    source_logits = _to_tensor(source_logits)
    source_labels = _to_tensor(source_labels).long()
    target_logits = _to_tensor(target_logits)
    
    source_scores = negentropy(source_logits)
    target_scores = negentropy(target_logits)
    sorted_source_scores, _ = torch.sort(source_scores)
    threshold = sorted_source_scores[-(source_logits.argmax(1) == source_labels).sum()]
    estimate = (target_scores > threshold).float().mean().item()
    return estimate


def predict_AC(source_logits, source_labels, target_logits):
    """Average Confidence."""
    target_logits = _to_tensor(target_logits)
    return torch.softmax(target_logits, dim=1).amax(1).mean().item()


def predict_DOC(source_logits, source_labels, target_logits):
    """Difference of Confidences."""
    source_logits = _to_tensor(source_logits)
    source_labels = _to_tensor(source_labels).long()
    target_logits = _to_tensor(target_logits)
    
    avg_source_conf = torch.softmax(source_logits, dim=1).amax(1).mean().item()
    avg_target_conf = torch.softmax(target_logits, dim=1).amax(1).mean().item()
    source_acc = (source_logits.argmax(1) == source_labels).float().mean().item()
    return source_acc + (avg_target_conf - avg_source_conf)


try:
    import ot
    
    def predict_COT(source_logits, source_labels, target_logits):
        """Confidence Optimal Transport."""
        source_logits = _to_tensor(source_logits)
        source_labels = _to_tensor(source_labels).long()
        target_logits = _to_tensor(target_logits)
        
        num_classes = source_logits.shape[1]
        source_label_dist = torch.nn.functional.one_hot(source_labels, num_classes).float().mean(0)
        target_probs = torch.softmax(target_logits, dim=1)
        
        cost_matrix = torch.stack([
            (target_probs - onehot).abs().sum(1)
            for onehot in torch.eye(num_classes, device=target_logits.device)
        ], dim=1) / 2
        
        # IMPORTANT: ot.emd() requires all arrays to be numpy
        uniform_dist = np.ones(len(target_probs)) / len(target_probs)
        source_dist = source_label_dist.cpu().numpy()
        cost_matrix_np = cost_matrix.cpu().numpy()
        
        ot_plan = ot.emd(uniform_dist, source_dist, cost_matrix_np)
        ot_cost = np.sum(ot_plan * cost_matrix_np)
        
        s_conf = torch.softmax(source_logits, dim=1).amax(1).mean().item()
        s_acc = (source_logits.argmax(1) == source_labels).float().mean().item()
        conf_gap = s_conf - s_acc
        err_est = ot_cost + conf_gap
        return 1. - err_est
    
    COT_AVAILABLE = True
    print("✓ COT available")
except ImportError:
    COT_AVAILABLE = False
    print("⚠ COT unavailable (install: pip install POT)")


BASELINE_METHODS = {
    'ATC': predict_ATC_maxconf,
    'ATC-NE': predict_ATC_negent,
    'AC': predict_AC,
    'DOC': predict_DOC,
}

if COT_AVAILABLE:
    BASELINE_METHODS['COT'] = predict_COT

print(f"✓ Baseline methods: {list(BASELINE_METHODS.keys())}")

⚠ COT unavailable (install: pip install POT)
✓ Baseline methods: ['ATC', 'ATC-NE', 'AC', 'DOC']


In [4]:
def collect_negative_scores(model, train_dataset, num_classes=10, device='cuda'):
    """Collect negative scores for decoy generation."""
    negative_scores_pools = {i: [] for i in range(num_classes)}
    
    model.eval()
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
    
    with torch.no_grad():
        for images, labels in tqdm(train_loader, desc='Collecting negative scores'):
            images, labels = images.to(device), labels.to(device)
            features = model.get_features(images)
            scores = model.linear2(F.relu(model.linear1(features)))
            
            for class_idx in range(num_classes):
                neg_mask = labels != class_idx
                if neg_mask.sum() > 0:
                    class_scores = scores[neg_mask, class_idx]
                    negative_scores_pools[class_idx].append(class_scores.cpu())
    
    for class_idx in range(num_classes):
        if negative_scores_pools[class_idx]:
            negative_scores_pools[class_idx] = torch.cat(negative_scores_pools[class_idx]).numpy()
        else:
            negative_scores_pools[class_idx] = np.array([])
    
    return negative_scores_pools



def create_score_feature_dataset(dataset, cnn_model, negative_scores_pools, device='cuda'):
    """Create dataset with scores and features."""
    cnn_scores_list, features_list, target_decoy_list, labels_list = [], [], [], []
    cnn_model.eval()
    
    with torch.no_grad():
        data_loader = DataLoader(dataset, batch_size=64, shuffle=False)
        for images, labels in tqdm(data_loader, desc='Creating score dataset', leave=False):
            images = images.to(device)
            features = cnn_model.get_features(images)
            scores = cnn_model.linear2(F.relu(cnn_model.linear1(features)))
            
            scores_cpu = scores.cpu()
            features_cpu = features.cpu()
            target_decoy_scores = scores_cpu.clone()
            
            for i, label in enumerate(labels):
                label_val = label.item()
                neg_pool = negative_scores_pools[label_val]
                if len(neg_pool) > 0:
                    random_neg_score = np.random.choice(neg_pool)
                    target_decoy_scores[i, label_val] = torch.tensor(random_neg_score, 
                                                                     dtype=target_decoy_scores.dtype)
            
            cnn_scores_list.append(scores_cpu)
            features_list.append(features_cpu)
            target_decoy_list.append(target_decoy_scores)
            labels_list.append(labels)
    
    class ScoreDataset:
        def __init__(self, scores, features, decoys, labels):
            self.cnn_scores = scores
            self.features = features
            self.target_decoy_scores = decoys
            self.labels = labels
        
        def __len__(self):
            return len(self.cnn_scores)
        
        def __getitem__(self, idx):
            return (self.cnn_scores[idx], self.features[idx], 
                    self.target_decoy_scores[idx], self.labels[idx])
    
    return ScoreDataset(torch.cat(cnn_scores_list), torch.cat(features_list),
                       torch.cat(target_decoy_list), torch.cat(labels_list))

In [5]:
def empirical_p_values(distribution, query):
    """Calculate empirical p-values from decoy distribution."""
    dist_len = len(distribution)
    p_values = np.zeros(len(query))
    sorted_dist = np.sort(distribution)
    for i, score in enumerate(query):
        p_values[i] = (dist_len - bisect(sorted_dist, score)) / dist_len
    return p_values


def estimate_pi0_storey(p_values, lambda_range=np.arange(0.05, 0.95, 0.05)):
    """Estimate proportion of nulls using Storey's method (LABEL-FREE)."""
    pi0_estimates = [np.mean(p_values > lam) / (1 - lam) for lam in lambda_range]
    return min(1.0, max(0, np.mean(pi0_estimates)))


def benjamini_hochberg(p_values, pi0=1.0):
    """Standard Benjamini-Hochberg procedure with π₀ adjustment."""
    n = len(p_values)
    sorted_idx = np.argsort(p_values)
    sorted_p = p_values[sorted_idx]
    q_values = np.zeros(n)
    
    for i in range(n):
        q = min(1.0, (n * sorted_p[i] * pi0) / (i + 1))
        q_values[i] = q
    
    # Enforce monotonicity
    for i in range(n-2, -1, -1):
        q_values[i] = min(q_values[i], q_values[i+1])
    
    # Map back to original order
    original_q = np.zeros(n)
    original_q[sorted_idx] = q_values
    return original_q


def calculate_fdr2_qvalues(target_scores, decoy_scores):
    """Calculate FDR2 q-values using TDC (LABEL-FREE)."""
    n = len(target_scores)
    combined_scores = np.maximum(target_scores, decoy_scores)
    is_target_win = (target_scores > decoy_scores).astype(int)
    
    sort_idx = np.argsort(combined_scores)[::-1]
    sorted_wins = is_target_win[sort_idx]
    
    fdr2_values = np.zeros(n)
    n_decoy_wins = 0
    
    for i in range(n):
        if sorted_wins[i] == 0:
            n_decoy_wins += 1
        fdr2_values[i] = (2 * n_decoy_wins) / (i + 1) if (i + 1) > 0 else 0
    
    # Enforce monotonicity
    q_fdr2 = np.minimum.accumulate(fdr2_values[::-1])[::-1]
    
    # Map back to original order
    final_q_fdr2 = np.zeros(n)
    for i in range(n):
        final_q_fdr2[sort_idx[i]] = q_fdr2[i]
    
    return final_q_fdr2


def control_fdr_mixmax(model_scores, target_labels, decoy_scores):
    """Mix-Max FDR control with FDR2, B-H, and Ground Truth."""
    n_samples = len(model_scores)
    df = pd.DataFrame({
        'original_index': np.arange(n_samples),
        'label': target_labels,
        'max_model_score': model_scores.max(axis=1),
        'max_decoy_score': decoy_scores.max(axis=1),
        'predicted_class': model_scores.argmax(axis=1)
    })
    df['max_score'] = df[['max_model_score', 'max_decoy_score']].max(axis=1)
    
    # Ground Truth FDR
    df_sorted_gt = df.sort_values(by='max_model_score', ascending=False).reset_index(drop=True)
    num_incorrect = 0
    fdr_gt_list = []
    for index, row in df_sorted_gt.iterrows():
        if row['predicted_class'] != row['label']:
            num_incorrect += 1
        fdr_gt_list.append(num_incorrect / (index + 1))
    q_values_gt = np.array(fdr_gt_list)
    for i in range(len(q_values_gt) - 2, -1, -1):
        q_values_gt[i] = min(q_values_gt[i], q_values_gt[i + 1])
    df_sorted_gt['q_values_ground_truth'] = q_values_gt
    gt_mapping = dict(zip(df_sorted_gt['original_index'], df_sorted_gt['q_values_ground_truth']))
    df['q_values_ground_truth'] = df['original_index'].map(gt_mapping)
    
    # FDR2 (TDC)
    target_scores = df['max_model_score'].values
    decoy_scores_vals = df['max_decoy_score'].values
    q_fdr2 = calculate_fdr2_qvalues(target_scores, decoy_scores_vals)
    df['q_values_fdr2'] = q_fdr2
    
    # Benjamini-Hochberg with Storey π₀
    p_values = empirical_p_values(decoy_scores_vals, target_scores)
    pi0 = estimate_pi0_storey(p_values)
    q_bh = benjamini_hochberg(p_values, pi0=pi0)
    df['q_values_bh'] = q_bh
    df['p_values'] = p_values
    df['pi0_storey'] = pi0
    
    return df


def control_fdr_binary_then_combine(model_scores, target_labels, decoy_scores, 
                                     method='min', use_bh=True):
    """Binary-then-Combine multi-class FDR control (LABEL-FREE)."""
    n_samples = len(model_scores)
    num_classes = model_scores.shape[1]
    class_results = []
    
    # Step 1: Binary FDR control for each class
    for class_k in range(num_classes):
        model_scores_k = model_scores[:, class_k]
        decoy_scores_k = decoy_scores[:, class_k]
        
        df_k = pd.DataFrame({
            'index': np.arange(n_samples),
            'model_score': model_scores_k,
            'decoy_score': decoy_scores_k,
            'class': class_k
        })
        
        if use_bh:
            p_values_k = empirical_p_values(decoy_scores_k, model_scores_k)
            pi0_k = estimate_pi0_storey(p_values_k)
            q_values_k = benjamini_hochberg(p_values_k, pi0=pi0_k)
            df_k['q_values_binary'] = q_values_k
            df_k['pi0_storey'] = pi0_k
        else:
            q_fdr2_k = calculate_fdr2_qvalues(model_scores_k, decoy_scores_k)
            df_k['q_values_binary'] = q_fdr2_k
        
        class_results.append(df_k)
    
    # Step 2: Combine q-values
    predicted_classes = model_scores.argmax(axis=1)
    
    if method == 'predicted':
        combined_q_values = np.ones(n_samples) * np.nan
        for sample_idx in range(n_samples):
            pred_class = predicted_classes[sample_idx]
            df_pred = class_results[pred_class]
            sample_row = df_pred[df_pred['index'] == sample_idx]
            if len(sample_row) > 0:
                combined_q_values[sample_idx] = sample_row['q_values_binary'].values[0]
    
    elif method == 'min':
        combined_q_values = np.ones(n_samples)
        for sample_idx in range(n_samples):
            min_q = 1.0
            for class_k in range(num_classes):
                df_k = class_results[class_k]
                sample_row = df_k[df_k['index'] == sample_idx]
                if len(sample_row) > 0:
                    q_val = sample_row['q_values_binary'].values[0]
                    if not np.isnan(q_val):
                        min_q = min(min_q, q_val)
            combined_q_values[sample_idx] = min_q
    
    # Step 3: Create final DataFrame
    df_final = pd.DataFrame({
        'index': np.arange(n_samples),
        'label': target_labels,
        'predicted_class': predicted_classes,
        'q_values_binary_combined': combined_q_values,
        'max_model_score': model_scores.max(axis=1)
    })
    
    # Add ground truth
    df_final_sorted = df_final.sort_values('max_model_score', ascending=False).reset_index(drop=True)
    num_incorrect = 0
    fdr_gt = []
    for idx, row in df_final_sorted.iterrows():
        if row['predicted_class'] != row['label']:
            num_incorrect += 1
        fdr_gt.append(num_incorrect / (idx + 1))
    q_gt = np.array(fdr_gt)
    for i in range(len(q_gt) - 2, -1, -1):
        q_gt[i] = min(q_gt[i], q_gt[i+1])
    df_final_sorted['q_values_ground_truth'] = q_gt
    gt_mapping = dict(zip(df_final_sorted['index'], df_final_sorted['q_values_ground_truth']))
    df_final['q_values_ground_truth'] = df_final['index'].map(gt_mapping)
    
    return df_final, class_results


print("✓ FDR control functions loaded")

✓ FDR control functions loaded


In [6]:
def estimate_pi0_storey_from_df(df, q_value_column):
    """Estimate π₀ from DataFrame (LABEL-FREE)."""
    if 'p_values' in df.columns:
        p_values = df['p_values'].dropna().values
        if len(p_values) > 0:
            lambda_range = np.arange(0.05, 0.95, 0.05)
            pi0_estimates = [np.mean(p_values > lam) / (1 - lam) for lam in lambda_range]
            pi0 = min(1.0, max(0, np.mean(pi0_estimates)))
            return pi0
    
    if 'pi0_storey' in df.columns:
        pi0 = df['pi0_storey'].iloc[0]
        if not np.isnan(pi0):
            return pi0
    
    # Fallback: empirical estimate
    score_column = 'max_model_score' if 'max_model_score' in df.columns else 'model_score'
    if score_column not in df.columns:
        return 0.5
    
    df_sorted = df.sort_values(by=score_column, ascending=False).reset_index(drop=True)
    top_k = max(100, int(0.2 * len(df_sorted)))
    top_samples = df_sorted.head(top_k)
    is_correct = (top_samples['predicted_class'] == top_samples['label']).astype(int)
    pi0_est = 1 - is_correct.mean()
    return np.clip(pi0_est, 0.01, 0.99)


def compute_ground_truth_curve(df):
    """Compute Ground Truth curve (uses labels - oracle)."""
    score_column = 'max_model_score' if 'max_model_score' in df.columns else 'model_score'
    if score_column not in df.columns:
        return pd.DataFrame({'q_value': [0.05], 'accuracy': [0.5]})
    
    df_gt = df.sort_values(by=score_column, ascending=False).reset_index(drop=True)
    
    if 'q_values_ground_truth' in df_gt.columns:
        df_gt = df_gt[~df_gt['q_values_ground_truth'].isna()].copy()
        q_col = 'q_values_ground_truth'
    else:
        df_gt['is_correct'] = (df_gt['predicted_class'] == df_gt['label']).astype(int)
        num_incorrect = 0
        q_values_gt = []
        for idx in range(len(df_gt)):
            if df_gt.loc[idx, 'is_correct'] == 0:
                num_incorrect += 1
            q_values_gt.append(num_incorrect / (idx + 1))
        df_gt['q_values_ground_truth'] = q_values_gt
        q_col = 'q_values_ground_truth'
    
    df_gt['is_correct'] = (df_gt['predicted_class'] == df_gt['label']).astype(int)
    df_gt['n_discoveries'] = np.arange(1, len(df_gt) + 1)
    df_gt['TP_true'] = df_gt['is_correct'].cumsum()
    df_gt['FP_true'] = df_gt['n_discoveries'] - df_gt['TP_true']
    
    total_correct = df_gt['is_correct'].sum()
    total_incorrect = len(df_gt) - total_correct
    total_samples = len(df)
    
    df_gt['TN_true'] = total_incorrect - df_gt['FP_true']
    df_gt['FN_true'] = total_correct - df_gt['TP_true']
    df_gt['Accuracy_true'] = (df_gt['TP_true'] + df_gt['TN_true']) / total_samples
    
    return df_gt[[q_col, 'Accuracy_true']].rename(
        columns={q_col: 'q_value_gt', 'Accuracy_true': 'accuracy_gt'})


def compute_method_estimation_curve(df, q_value_column, pi0):
    """Compute LABEL-FREE estimation curve."""
    total_samples = len(df)
    df_method = df[~df[q_value_column].isna()].copy()
    
    if len(df_method) == 0:
        return pd.DataFrame()
    
    df_method = df_method.sort_values(by=q_value_column, ascending=True).reset_index(drop=True)
    df_method['n_discoveries'] = np.arange(1, len(df_method) + 1)
    
    # Label-free estimation
    df_method['FP_est'] = df_method['n_discoveries'] * df_method[q_value_column]
    df_method['TP_est'] = df_method['n_discoveries'] - df_method['FP_est']
    df_method['TN_est'] = total_samples * pi0 - df_method['FP_est']
    df_method['FN_est'] = total_samples * (1 - pi0) - df_method['TP_est']
    
    df_method['FP_est'] = np.maximum(0, df_method['FP_est'])
    df_method['TP_est'] = np.maximum(0, df_method['TP_est'])
    df_method['TN_est'] = np.maximum(0, df_method['TN_est'])
    df_method['FN_est'] = np.maximum(0, df_method['FN_est'])
    
    df_method['Accuracy_est'] = (df_method['TP_est'] + df_method['TN_est']) / total_samples
    
    # True metrics (for comparison)
    df_method['is_correct'] = (df_method['predicted_class'] == df_method['label']).astype(int)
    df_method['TP_true'] = df_method['is_correct'].cumsum()
    df_method['FP_true'] = df_method['n_discoveries'] - df_method['TP_true']
    
    total_correct = df_method['is_correct'].sum()
    total_incorrect = len(df_method) - total_correct
    
    df_method['TN_true'] = total_incorrect - df_method['FP_true']
    df_method['FN_true'] = total_correct - df_method['TP_true']
    df_method['Accuracy_true'] = (df_method['TP_true'] + df_method['TN_true']) / total_samples
    
    return df_method[[q_value_column, 'Accuracy_est', 'Accuracy_true',
                      'FP_est', 'TP_est', 'FN_est', 'TN_est',
                      'FP_true', 'TP_true', 'FN_true', 'TN_true']].rename(
        columns={q_value_column: 'q_value_method'})


print("✓ Accuracy estimation functions loaded")

✓ Accuracy estimation functions loaded


In [7]:
def plot_accuracy_estimation(df, q_value_column, method_name, corruption_name, save_dir):
    """Plot accuracy estimation curve (publication quality, 6x4 inches)."""
    os.makedirs(save_dir, exist_ok=True)
    
    pi0 = estimate_pi0_storey_from_df(df, q_value_column)
    df_gt = compute_ground_truth_curve(df)
    df_method = compute_method_estimation_curve(df, q_value_column, pi0)
    
    if len(df_method) == 0:
        print(f"  No valid data for {corruption_name}")
        return None, pi0
    
    fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
    
    # Plot method estimation
    ax.plot(df_method['q_value_method'], df_method['Accuracy_est'],
            color='#FF9800', linewidth=2.5, label=f'{method_name}', alpha=0.9)
    
    # Plot ground truth
    ax.plot(df_gt['q_value_gt'], df_gt['accuracy_gt'],
            color='black', linewidth=2.5, linestyle='--', label='Ground Truth', alpha=0.7)
    
    # FDR reference lines
    for fdr in [0.01, 0.05, 0.1]:
        if fdr <= 0.5:
            ax.axvline(fdr, color='gray', linestyle=':', alpha=0.4, linewidth=1.5)
            ax.text(fdr, 0.02, f'{fdr:.2f}', rotation=0, va='bottom', ha='center',
                   fontsize=10, alpha=0.7)
    
    ax.set_xlabel('Q-value (FDR)', fontweight='bold')
    ax.set_ylabel('Accuracy', fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.set_xlim(0, min(0.5, max(df_method['q_value_method'].max(), df_gt['q_value_gt'].max())))
    
    ax.legend(loc='lower left', frameon=True, framealpha=0.9)
    ax.grid(True, alpha=0.3, linestyle='--')
    
    corruption_title = corruption_name.replace('_', ' ').title()
    ax.set_title(f'{corruption_title}', fontweight='bold')
    
    filename = f'{save_dir}/accuracy_{corruption_name}'
    plt.savefig(filename + '.png', dpi=300, bbox_inches='tight')
    plt.savefig(filename + '.pdf', bbox_inches='tight')
    plt.close()
    
    print(f"  ✓ {corruption_name}: π₀={pi0:.3f}, max_acc={df_method['Accuracy_est'].max():.3f}")
    return df_method, pi0


def plot_fdr_curve(df, corruption_name, save_dir):
    """Plot FDR control curve (publication quality, 6x4 inches)."""
    os.makedirs(save_dir, exist_ok=True)
    
    fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
    
    methods = [
        ('q_values_ground_truth', 'Ground Truth', '#2E7D32', '--', 2.5),
        ('q_values_fdr2', 'FDR2 (TDC)', '#F57C00', '-', 2.0),
        ('q_values_bh', 'Benjamini-Hochberg', '#7B1FA2', '-', 2.0),
        ('q_values_binary_combined', 'BC-Min (BH)', '#FF9800', '-', 2.5)
    ]
    
    for col_name, label, color, linestyle, linewidth in methods:
        if col_name not in df.columns:
            continue
        
        df_plot = df[~df[col_name].isna()].copy()
        if len(df_plot) == 0:
            continue
        
        df_sorted = df_plot.sort_values(col_name).reset_index(drop=True)
        q_vals = df_sorted[col_name].values
        n_discoveries = np.arange(1, len(df_sorted) + 1)
        
        ax.plot(q_vals, n_discoveries, label=label, linewidth=linewidth,
                color=color, linestyle=linestyle, alpha=0.8)
    
    # FDR reference lines
    for fdr in [0.01, 0.05, 0.1]:
        if fdr <= 0.2:
            ax.axvline(fdr, color='gray', linestyle=':', alpha=0.4, linewidth=1)
            ax.text(fdr, ax.get_ylim()[1]*0.95, f'{fdr:.2f}',
                   rotation=90, va='top', ha='right', fontsize=9, alpha=0.7)
    
    ax.set_xlabel('Q-value (FDR)', fontweight='bold')
    ax.set_ylabel('Number of Discoveries', fontweight='bold')
    ax.set_xlim(0, 0.2)
    ax.legend(fontsize=9, framealpha=0.9, loc='lower right')
    ax.grid(alpha=0.3, linestyle='--')
    
    corruption_title = corruption_name.replace('_', ' ').title()
    ax.set_title(f'{corruption_title}', fontweight='bold')
    
    filename = f'{save_dir}/fdr_{corruption_name}'
    plt.savefig(filename + '.png', dpi=300, bbox_inches='tight')
    plt.savefig(filename + '.pdf', bbox_inches='tight')
    plt.close()
    
    print(f"  ✓ {corruption_name}")


print("✓ Plotting functions loaded")

✓ Plotting functions loaded


In [8]:
def compute_mano_score(logits: torch.Tensor, p: int = 4, eta: float = 5.0) -> float:
    """Compute MANO score for a batch of logits."""
    N, K = logits.shape
    
    # Calibration criterion
    softmax_probs = torch.softmax(logits, dim=1)
    criterion = -torch.mean(torch.log(softmax_probs))
    
    # Normalization
    if criterion <= eta:
        # Taylor approximation
        normalized = 1 + logits + (logits ** 2) / 2
        normalized = normalized - normalized.min(dim=1, keepdim=True)[0]
        normalized = normalized / normalized.sum(dim=1, keepdim=True)
    else:
        # Softmax
        normalized = softmax_probs
    
    # Lp norm
    lp_norm = torch.sum(torch.abs(normalized) ** p) ** (1 / p)
    mano_score = lp_norm / ((p * N * K) ** (1 / p))
    
    return mano_score.item()


def compute_accuracy_from_logits(predictions: torch.Tensor, labels: torch.Tensor) -> float:
    """Compute classification accuracy."""
    pred_classes = torch.argmax(predictions, dim=1)
    correct = (pred_classes == labels).sum().item()
    total = labels.size(0)
    return 100.0 * correct / total


def plot_mano_vs_accuracy(data_dict: Dict[str, Dict[str, torch.Tensor]],
                          save_path: str,
                          title: str = "MANO vs Test Accuracy - CIFAR-10-C"):
    """Create MANO vs Accuracy scatter plot (publication quality, 6x4 inches)."""
    mano_scores = []
    accuracies = []
    condition_names = []
    
    # Compute scores
    for condition, data in data_dict.items():
        logits = data['logits']
        labels = data['labels']
        
        mano_score = compute_mano_score(logits)
        accuracy = compute_accuracy_from_logits(logits, labels)
        
        mano_scores.append(mano_score)
        accuracies.append(accuracy)
        condition_names.append(condition)
        
        print(f"  {condition}: MANO={mano_score:.4f}, Acc={accuracy:.2f}%")
    
    # Create plot
    fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
    
    scatter = ax.scatter(mano_scores, accuracies,
                        s=100, alpha=0.7,
                        c=range(len(mano_scores)),
                        cmap='viridis',
                        edgecolors='black',
                        linewidth=1)
    
    # Linear regression
    if len(mano_scores) > 1 and np.std(mano_scores) > 1e-6 and np.std(accuracies) > 1e-6:
        z = np.polyfit(mano_scores, accuracies, 1)
        p = np.poly1d(z)
        x_line = np.linspace(min(mano_scores), max(mano_scores), 100)
        ax.plot(x_line, p(x_line), "r--", alpha=0.8, linewidth=2, label='Linear fit')
        
        correlation = np.corrcoef(mano_scores, accuracies)[0, 1]
        r_squared = correlation ** 2
        rho, _ = spearmanr(mano_scores, accuracies)
        
        ax.text(0.05, 0.95,
                f'$R^2$ = {r_squared:.3f}\n$\\rho$ = {rho:.3f}',
                transform=ax.transAxes,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
                fontsize=11)
    
    ax.set_xlabel('MANO Score', fontweight='bold')
    ax.set_ylabel('Test Accuracy (%)', fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    plt.savefig(save_path.replace('.png', '') + '.png', dpi=300, bbox_inches='tight')
    plt.savefig(save_path.replace('.png', '') + '.pdf', bbox_inches='tight')
    plt.close()
    
    print(f"\n✓ MANO plot saved")
    return mano_scores, accuracies, condition_names


print("✓ MANO functions loaded")

✓ MANO functions loaded


In [9]:
def plot_method_comparison(results_df, corruption_name, save_dir='figures/comparison'):
    """
    Compare all methods (FDR + Baselines) for a single corruption.
    Publication quality 6x4 inches.
    """
    os.makedirs(save_dir, exist_ok=True)
    
    fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
    
    # Filter for this corruption
    df_corr = results_df[results_df['corruption'] == corruption_name]
    
    # Get ground truth
    true_acc = df_corr['true_accuracy'].iloc[0]
    
    # Sort methods by estimate
    df_sorted = df_corr.sort_values('estimated_accuracy', ascending=False)
    
    methods = df_sorted['method'].values
    estimates = df_sorted['estimated_accuracy'].values
    errors = np.abs(estimates - true_acc)
    
    # Color mapping
    colors = []
    for method in methods:
        if 'BC-Min' in method:
            colors.append('#FF9800')  # Orange - our method
        elif method in ['FDR2', 'B-H']:
            colors.append('#7B1FA2')  # Purple - other FDR
        else:
            colors.append('#1976D2')  # Blue - baselines
    
    # Bar plot
    x_pos = np.arange(len(methods))
    bars = ax.bar(x_pos, estimates, color=colors, alpha=0.7, edgecolor='black', linewidth=1)
    
    # Ground truth line
    ax.axhline(true_acc, color='red', linestyle='--', linewidth=2.5, 
               label=f'True Accuracy ({true_acc:.3f})', alpha=0.8)
    
    # Add error annotations on bars
    for i, (bar, err) in enumerate(zip(bars, errors)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{err:.3f}',
                ha='center', va='bottom', fontsize=8, rotation=0)
    
    ax.set_xlabel('Method', fontweight='bold')
    ax.set_ylabel('Estimated Accuracy', fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(methods, rotation=45, ha='right', fontsize=10)
    ax.set_ylim(0, 1.0)
    ax.legend(loc='lower right', framealpha=0.9)
    ax.grid(axis='y', alpha=0.3)
    
    corruption_title = corruption_name.replace('_', ' ').title()
    ax.set_title(f'Method Comparison: {corruption_title}', fontweight='bold')
    
    filename = f'{save_dir}/comparison_{corruption_name}'
    plt.savefig(filename + '.png', dpi=300, bbox_inches='tight')
    plt.savefig(filename + '.pdf', bbox_inches='tight')
    plt.close()
    
    print(f"  ✓ {corruption_name}")


def plot_overall_comparison(results_df, save_dir='figures/comparison'):
    """
    Overall comparison across all corruptions.
    Shows mean absolute error for each method.
    """
    os.makedirs(save_dir, exist_ok=True)
    
    # Compute MAE for each method
    mae_data = []
    for method in results_df['method'].unique():
        df_method = results_df[results_df['method'] == method]
        mae = np.abs(df_method['estimated_accuracy'] - df_method['true_accuracy']).mean()
        mae_data.append({'Method': method, 'MAE': mae})
    
    mae_df = pd.DataFrame(mae_data).sort_values('MAE')
    
    fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
    
    # Color by method type
    colors = []
    for method in mae_df['Method']:
        if 'BC-Min' in method:
            colors.append('#FF9800')
        elif method in ['FDR2', 'B-H']:
            colors.append('#7B1FA2')
        else:
            colors.append('#1976D2')
    
    x_pos = np.arange(len(mae_df))
    ax.bar(x_pos, mae_df['MAE'], color=colors, alpha=0.7, edgecolor='black', linewidth=1)
    
    ax.set_xlabel('Method', fontweight='bold')
    ax.set_ylabel('Mean Absolute Error', fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(mae_df['Method'], rotation=45, ha='right')
    ax.set_title('Overall Performance Across All Corruptions', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for i, (idx, row) in enumerate(mae_df.iterrows()):
        ax.text(i, row['MAE'] + 0.005, f"{row['MAE']:.4f}",
                ha='center', va='bottom', fontsize=9)
    
    filename = f'{save_dir}/overall_comparison'
    plt.savefig(filename + '.png', dpi=300, bbox_inches='tight')
    plt.savefig(filename + '.pdf', bbox_inches='tight')
    plt.close()
    
    print("\n✓ Overall comparison plot saved")
    print("\nMean Absolute Error (MAE) by Method:")
    print(mae_df.to_string(index=False))
    
    return mae_df


print("✓ Comparison plotting functions loaded")

✓ Comparison plotting functions loaded


In [ ]:
PATH_DATA = '../../data/BCSS/training/bcss.medium.training.torch'


data = torch.load(PATH_DATA)
train_ds = create_score_feature_dataset_bcss(data, DEVICE)
print('Train loaded')

print("Training separate flows for each class...")
separate_flows = SeparateClassFlows(num_classes=NUM_CLASSES, n_flows=10, feature_dim=64, hidden_dim=64).to(DEVICE)
separate_flows = separate_flows.train_separate(train_ds, epochs=5, lr=1e-4, device=DEVICE)
print('here trained')
torch.save(separate_flows.state_dict(), 'BCSS/bcss_cond_flows_model.pth')
separate_flows.load_state_dict(torch.load('BCSS/bcss_cond_flows_model.pth'))
print('uploaded')

